# Customer experience EDA
Tài liệu này tập trung phân tích trải nghiệm khách hàng thông qua dữ liệu trả hàng và đánh giá sản phẩm.

### Prepare libraries and load data:

In [40]:
import pandas as pd
import seaborn as sns
import matplotlib as plt

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

df_pduct = pd.read_parquet('../dataset/cleaned/products.parquet')
df_cus = pd.read_parquet('../dataset/cleaned/customers.parquet')
df_ord = pd.read_parquet('../dataset/cleaned/orders.parquet')
df_ret = pd.read_parquet('../dataset/cleaned/returns.parquet')
df_rev = pd.read_parquet('../dataset/cleaned/reviews.parquet')
df_rev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113551 entries, 0 to 113550
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   review_id     113551 non-null  string        
 1   order_id      113551 non-null  string        
 2   product_id    113551 non-null  string        
 3   customer_id   113551 non-null  string        
 4   review_date   113551 non-null  datetime64[ns]
 5   rating        113551 non-null  int64         
 6   review_title  113551 non-null  category      
dtypes: category(1), datetime64[ns](1), int64(1), string(4)
memory usage: 5.3 MB


In [41]:
df_pduct = df_pduct.drop(columns=['price', 'cogs', 'product_name'])
df_cus = df_cus.drop(columns=['signup_date', 'gender', 'zip'])
df_ord = df_ord.drop(columns=['zip'])
df_ret = df_ret.drop(columns=['product_id', 'return_id', 'return_date', 'return_quantity'])
df_rev = df_rev.drop(columns=['product_id', 'customer_id', 'review_id', 'review_date'])

df_master = df_ord.merge(df_cus, on='customer_id', how='left')
df_master = df_master.merge(df_ret, on='order_id', how='left')
df_master = df_master.merge(df_rev, on='order_id', how='left')
# df_master = df_master.merge(df_pduct, on='product_id', how='left')

df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 653004 entries, 0 to 653003
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             653004 non-null  string        
 1   order_date           653004 non-null  datetime64[ns]
 2   customer_id          653004 non-null  string        
 3   order_status         653004 non-null  category      
 4   payment_method       653004 non-null  category      
 5   device_type          653004 non-null  category      
 6   order_source         653004 non-null  category      
 7   payment_value        653004 non-null  float64       
 8   installments         653004 non-null  int64         
 9   city                 653004 non-null  category      
 10  age_group            653004 non-null  category      
 11  acquisition_channel  653004 non-null  category      
 12  return_reason        39939 non-null   category      
 13  refund_amount 

In [39]:
for col in df_master.columns:
    if df_master[col].isna().any():
        if pd.api.types.is_float_dtype(df_master[col]):
            df_master[col] = df_master[col].fillna(0.0)
        elif df_master[col].dtype.name == 'category':
            df_master[col] = df_master[col].cat.add_categories('None')
            df_master[col] = df_master[col].fillna('None')
df_master.info()
display(df_master.sample(15))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 653004 entries, 0 to 653003
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             653004 non-null  string        
 1   order_date           653004 non-null  datetime64[ns]
 2   customer_id          653004 non-null  string        
 3   order_status         653004 non-null  category      
 4   payment_method       653004 non-null  category      
 5   device_type          653004 non-null  category      
 6   order_source         653004 non-null  category      
 7   payment_value        653004 non-null  float64       
 8   installments         653004 non-null  int64         
 9   city                 653004 non-null  category      
 10  age_group            653004 non-null  category      
 11  acquisition_channel  653004 non-null  category      
 12  return_reason        653004 non-null  category      
 13  refund_amount 

,order_id,order_date,customer_id,order_status,payment_method,device_type,order_source,payment_value,installments,city,age_group,acquisition_channel,return_reason,refund_amount,rating,review_title
284790,363034,2016-03-03,144261,delivered,apple_pay,desktop,referral,24518.37,3,Ho Chi Minh City,25-34,social_media,None,0.0,0.0,None
419165,534674,2017-09-19,4806,delivered,credit_card,mobile,paid_search,12921.17,1,Nam Dinh,25-34,paid_search,None,0.0,0.0,None
155996,198864,2014-07-04,133291,delivered,credit_card,tablet,social_media,16729.57,3,Rach Gia,55+,social_media,None,0.0,0.0,None
47465,60447,2013-04-02,3414,delivered,paypal,desktop,direct,39746.48,6,Uong Bi,18-24,organic_search,None,0.0,0.0,None
533367,680874,2019-08-10,121417,delivered,credit_card,mobile,email_campaign,7741.53,3,Quy Nhon,25-34,paid_search,None,0.0,4.0,Good overall
68505,87225,2013-06-14,115489,delivered,paypal,mobile,referral,66990.06,3,Dong Hoi,55+,organic_search,None,0.0,2.0,Some issues
576754,736340,2020-10-10,72838,delivered,credit_card,desktop,email_campaign,81224.10,12,Cam Pha,35-44,paid_search,None,0.0,5.0,Excellent product!
312842,398910,2016-06-01,132121,delivered,credit_card,mobile,social_media,28337.24,3,Tuy Hoa,25-34,organic_search,None,0.0,4.0,Solid choice
29357,37448,2012-12-22,63554,delivered,credit_card,desktop,referral,4511.36,1,Bac Giang,45-54,direct,None,0.0,0.0,None
410802,524139,2017-08-07,43233,delivered,credit_card,mobile,organic_search,43190.56,1,Thai Nguyen,45-54,social_media,None,0.0,5.0,Excellent product!
